# 04｜連續性與中間值定理：從「一定存在」到「把它找出來」

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/calculus-for-iih/blob/main/notebooks/04_continuity_ivt.ipynb)

臺北醫學大學 智慧醫療跨領域　微積分（1151）　**自由練習，不計分、不用繳交**

對應課堂投影片：**Calculus04 連續性與中間值定理**

**怎麼用**
1. 按上方選單「執行階段 → 全部執行」，從上往下看圖和數字。
2. 看到 **✏️ 動手** 的格子，就自己改改看。填空沒填也不會出錯，程式會提醒你「還沒填」。
3. 卡住就點開「💡 提示」，真的不行再看「參考答案」。

不需要先會寫 Python，每一段程式前面都有中文說明。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(ok, hint):
    """檢查答案：對了顯示 ✓，不對顯示提示。"""
    print("✓ 正確！" if ok else "✗ 還沒對，" + hint)

## 1. 三種不連續

函數在一點連續，要同時滿足三件事：**有定義、極限存在、兩者相等**。下面三張圖各壞掉一個地方。

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))

x = np.linspace(-2, 2, 400)
ax[0].plot(x, x + 1, color="#028090", lw=3)
ax[0].plot([0], [1], "o", mfc="white", mec="#028090", ms=9)
ax[0].plot([0], [2.2], "o", color="#028090", ms=8)
ax[0].set_title("removable")

left, right = x[x < 0], x[x >= 0]
ax[1].plot(left, left + 1, color="#028090", lw=3)
ax[1].plot(right, right + 2.5, color="#028090", lw=3)
ax[1].set_title("jump")

xl, xr = np.linspace(-2, -0.15, 200), np.linspace(0.15, 2, 200)
ax[2].plot(xl, 1 / xl, color="#028090", lw=3)
ax[2].plot(xr, 1 / xr, color="#028090", lw=3)
ax[2].set_title("infinite")

for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. ✏️ 動手 1：臨床上的跳躍不連續

依 eGFR 調整劑量是一個分段函數（示意規則），在級距的邊界會**跳**。改 `a`、`b` 看看差一點點時，係數差多少。

In [ ]:
def dose_factor(e):
    return 1.0 if e >= 60 else 0.75 if e >= 30 else 0.5

a, b = 59, 61      # ← 改這裡，例如 29 和 31

print(f"eGFR {a}：係數 {dose_factor(a)}")
print(f"eGFR {b}：係數 {dose_factor(b)}")

e = np.linspace(15, 90, 600)
plt.figure(figsize=(7, 3))
plt.plot(e, [dose_factor(v) for v in e], ".", color="#028090", ms=3)
plt.xlabel("eGFR"); plt.ylabel("dose factor"); plt.grid(alpha=0.3); plt.show()

eGFR 是估算值，兩次檢驗可能相差幾個單位，所以邊界附近的病人不能只看單次數值套表。

## 3. 中間值定理：一定有根

$f(x)=x^3+x-1$ 是多項式，所以連續。$f(0)=-1<0$、$f(1)=1>0$，曲線要從負到正、又不能跳，就一定會穿過 0。

In [ ]:
f = lambda x: x**3 + x - 1
x = np.linspace(0, 1, 200)

plt.figure(figsize=(6, 3.5))
plt.plot(x, f(x), color="#028090", lw=3)
plt.axhline(0, color="gray", lw=1)
plt.plot([0, 1], [f(0), f(1)], "o", color="#D9534F")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.grid(alpha=0.3); plt.show()

## 4. 二分法：把定理變成演算法

定理只說根存在，二分法把它找出來：每次取中點，保留**兩端異號**的那半邊，區間就砍半。下面印出每一步。

In [ ]:
a, b = 0, 1
print("步驟      a         b         中點 m     f(m)")
for i in range(1, 11):
    m = (a + b) / 2
    print(f"{i:>3}   {a:.5f}   {b:.5f}   {m:.5f}   {f(m):+.5f}")
    if f(a) * f(m) <= 0:
        b = m      # 根在左半邊
    else:
        a = m      # 根在右半邊
print("根約為", round((a + b) / 2, 4))

10 步後區間只剩 $1/2^{10}\approx0.001$ 寬，每一步都靠中間值定理保證「根還在裡面」。

## 5. ✏️ 動手 2：自己補二分法

只差一行：中點怎麼算？

In [ ]:
def bisection(f, a, b, tol=1e-6):
    if f(a) * f(b) > 0:
        raise ValueError("f(a)、f(b) 必須異號")
    while b - a > tol:
        m = None      # ✏️ 填空：a 和 b 的中點
        if m is None:
            return None
        if f(a) * f(m) <= 0:
            b = m
        else:
            a = m
    return (a + b) / 2

# ---- 自動檢查（不用改）----
root = bisection(lambda x: x**3 + x - 1, 0, 1)
if root is None:
    print("還沒填，把 None 換成中點的式子。")
else:
    check(abs(root - 0.682328) < 1e-4, "中點是兩端相加除以 2。")
    print("根 ≈", round(root, 6))

<details><summary>參考答案</summary>

`m = (a + b) / 2`
</details>

下面幾節要用到二分法，所以老師另外提供一份 `solve`，你沒填上面那格也能繼續往下跑。

In [ ]:
def solve(f, a, b, tol=1e-6):
    "老師提供的二分法。"
    while b - a > tol:
        m = (a + b) / 2
        if f(a) * f(m) <= 0:
            b = m
        else:
            a = m
    return (a + b) / 2

## 6. ✏️ 動手 3：藥效何時開始、何時結束？

口服藥濃度模型（示意）：$C(t)=20(e^{-0.2t}-e^{-t})$，最低有效濃度 MEC = 6 mg/L。

同一條曲線用兩次中間值定理：上升段一定有「起效時刻」、下降段一定有「失效時刻」。

In [ ]:
C = lambda t: 20 * (np.exp(-0.2 * t) - np.exp(-t))
MEC = 6
g = lambda t: C(t) - MEC        # 令 g(t) = 0，就是濃度剛好等於 MEC

t_on = solve(g, 0, 2)           # 上升段
t_off = solve(g, 2, 12)         # 下降段
print(f"起效 t1 ≈ {t_on:.2f} 小時，失效 t2 ≈ {t_off:.2f} 小時")

t = np.linspace(0, 12, 300)
plt.figure(figsize=(7, 3.5))
plt.plot(t, C(t), color="#028090", lw=3)
plt.axhline(MEC, color="#D9534F", ls="--")
plt.plot([t_on, t_off], [MEC, MEC], "ko")
plt.xlabel("t (hours)"); plt.ylabel("C (mg/L)"); plt.grid(alpha=0.3); plt.show()

answer_3 = None      # ✏️ 填空：有效作用時間是幾小時？（用 t_on、t_off 寫出來）

# ---- 自動檢查（不用改）----
if answer_3 is None:
    print("還沒填，有效作用時間 = 失效時刻 − 起效時刻。")
else:
    check(abs(answer_3 - (t_off - t_on)) < 0.01, "是 t_off 減 t_on。")

## 7. 虛擬病人：每個人的代謝速度不同

下面產生 8 位**虛擬**病人，消除速率 $k$ 各不相同，再用同一套方法算出有效作用時間。程式寫好一次，就能處理很多人。

In [ ]:
rng = np.random.default_rng(42)
ks = rng.uniform(0.12, 0.30, 8).round(3)

print("病人    k      有效作用時間（小時）")
for i, k in enumerate(ks, 1):
    c = lambda t: 20 * (np.exp(-k * t) - np.exp(-t)) - MEC
    print(f"V{i:02d}   {k:.3f}   {solve(c, 2, 30) - solve(c, 0, 2):.1f}")

k 越小（代謝越慢），藥效維持越久。這就是「腎功能差的人要調整給藥」的數學直覺。

## 8. 用程式檢查 AI 的答案

> AI 說：$f(x)=1/x$，$f(-1)<0$、$f(1)>0$，由勘根定理，$f(x)=0$ 在 $(-1,1)$ 內有根。

如果直接用二分法，會得到什麼？

In [ ]:
r = solve(lambda x: 1 / x if x != 0 else np.inf, -1, 1.1)
print("二分法停在 x ≈", abs(round(r, 6)))
print("但是 1/x 在那附近的大小約為", round(abs(1 / r)))

二分法停在 0 附近，但那裡的函數值不是 0，絕對值反而非常大。因為 $1/x$ 在 $x=0$ **不連續**，定理的條件不成立，AI 的結論是錯的，$1/x$ 根本沒有根。

**用定理前先檢查條件；程式也會在條件不成立時給出看似合理的數字。**

## 小結

- 連續 = 有定義、極限存在、兩者相等。
- 中間值定理保證存在，二分法負責找出來。
- 定理的條件不能省，程式也不會幫你檢查。

---
*本 notebook 為教學練習，數值與情境皆為示意，資料為程式產生的虛擬資料，非真實病人資料，也不是用藥建議。*